In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
from matplotlib import pyplot as plt
import statsmodels.formula.api as smf
pd.set_option('display.max_rows',200)
pd.set_option('display.max_columns', 100)
from statsmodels.stats.multitest import multipletests
%matplotlib inline

from matplotlib import cm, colors

from nilearn import datasets
from nilearn import plotting as nlp
from scipy import stats
from matplotlib.gridspec import GridSpec

from dl_morphometrics_helpers import constants as cfg


In [ ]:
cp = sns.color_palette()
global_df = pd.read_csv('../data/derivatives/fs_stats/recon-all/BrainConfounds.tsv', sep='\t')

global_df_ct1 = pd.read_csv('../data/derivatives/fs_stats/recon-all_clinical_t1/BrainConfounds.tsv', sep='\t')
global_df_ct2 = pd.read_csv('../data/derivatives/fs_stats/recon-all_clinical_t2/BrainConfounds.tsv', sep='\t')
global_df_not2 = pd.read_csv('../data/derivatives/fs_stats/recon-all_not2/BrainConfounds.tsv', sep='\t')

global_df_ct1r5 = pd.read_csv('../data/derivatives/fs_stats/recon-all_clinical_t1_resample-5/BrainConfounds.tsv', sep='\t')
global_df_ct1r3 = pd.read_csv('../data/derivatives/fs_stats/recon-all_clinical_t1_resample-3/BrainConfounds.tsv', sep='\t')

In [ ]:
measures = ['Subcortical gray matter volume',
       'Total gray matter volume', 'Total cortical gray matter volume',
       'Mask Volume', 'Brain Segmentation Volume Without Ventricles',
       'Left hemisphere cerebral white matter volume',
       'Right hemisphere cerebral white matter volume',
       'Total cerebral white matter volume']
ct2_measures = [mm + '_ract2' for mm in measures]
global_df_ct2.columns = ['subject', 'session',] + ct2_measures

not2_measures = [mm + '_ranot2' for mm in measures]
global_df_not2.columns = ['subject', 'session',] + not2_measures

ct1r3_measures = [mm + '_ract1r3' for mm in measures]
global_df_ct1r3.columns = ['subject', 'session',] + ct1r3_measures

ct1r5_measures = [mm + '_ract1r5' for mm in measures]
global_df_ct1r5.columns = ['subject', 'session',] + ct1r5_measures

In [ ]:
global_df = global_df.merge(global_df_ct1, how='inner', on=['subject', 'session'], suffixes=['_ra', '_ract1'])
global_df = global_df.merge(global_df_ct2, how='inner', on=['subject', 'session'])
global_df = global_df.merge(global_df_not2, how='inner', on=['subject', 'session'])
global_df = global_df.merge(global_df_ct1r3, how='left', on=['subject', 'session'])
global_df = global_df.merge(global_df_ct1r5, how='left', on=['subject', 'session'])


In [ ]:
global_df.iloc[:, 2:] /= 10000

In [ ]:
ses_info = pd.read_csv('/data/ABCD_DSST/ABCD/imaging_data/fast_track/sessions.tsv', sep='\t')

In [ ]:
ses_info['subject'] = ses_info.participant_id.str.split('-').str[1]
ses_info['session'] = ses_info.session_id.str.split('-').str[1]
ses_info['age_years'] = ses_info.age / 12
pml = len(global_df)
global_df = global_df.merge(ses_info.loc[:, ['subject', 'session', 'age_years']], how='left', on=['subject', 'session'])
assert pml == len(global_df)

In [ ]:
sex_df = ses_info.query("sex.notnull()").groupby('subject').sex.first().reset_index()
global_df = global_df.merge(sex_df, how='left', on='subject')


# get ages for last wave

In [ ]:

fastages = pd.read_csv('/data/ABCD_DSST/ABCD/imaging_data/fast_track/code/abcd_fastqc01_history/2024-05-01/abcd_fastqc01_ages_innerjoin_sessions_without_age.tsv', sep='\t')
fastages['subject'] = fastages.participant_id.str.split('-').str[1]
fastages['session'] = fastages.session_id.str.split('-').str[1]
fastages['age_years'] = fastages.age / 12


In [ ]:
global_df = global_df.merge(fastages.loc[:, ['subject', 'session', 'age_years']], how='left', on=['subject', 'session'], suffixes=['', '_fa'])
global_df['age_years'] = global_df.age_years.fillna(global_df.age_years_fa)
assert len(global_df.loc[global_df.age_years.isnull()]) == 0

In [ ]:
cfg.metric_cols = ['Subcortical gray matter volume',
       'Total gray matter volume', 'Total cortical gray matter volume',
       'Mask Volume', 'Brain Segmentation Volume Without Ventricles',
       'Left hemisphere cerebral white matter volume',
       'Right hemisphere cerebral white matter volume',
       'Total cerebral white matter volume']

In [ ]:
dif_cols_ct1 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1'
    dif_cols_ct1.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

dif_cols_ct2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct2'
    dif_cols_ct2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract2'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

dif_cols_not2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_not2'
    dif_cols_not2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ranot2'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

dif_cols_ct1r3 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1r3'
    dif_cols_ct1r3.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1r3'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

dif_cols_ct1r5 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1r5'
    dif_cols_ct1r5.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1r5'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

dif_cols_ct1ct2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ct1_dif_ct2'
    dif_cols_ct1ct2.append(dif_col)
    ra_col = f'{mc}_ract1'
    rac_col = f'{mc}_ract2'
    global_df[dif_col] = ((global_df[rac_col] - global_df[ra_col]) / global_df[ra_col] ) * 100

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_ct1'], '.', label='Cereb GMV CT1')
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_ct1r5'], '.', label='Cort GMV CT1R5')

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of rac from ra')
ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_pct_ra_dif_ct1'], '.', label='Subcort GMV')
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_ct1'], '.', label='Cort GMV')
ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_pct_ra_dif_ct1'], '.', label='Cereb WMV')
ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of rac from ra')
ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_pct_ra_dif_not2'], '.', label='Subcort GMV')
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_not2'], '.', label='Cort GMV')
ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_pct_ra_dif_not2'], '.', label='Cereb WMV')
ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of ranot2 from ra')
ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_pct_ra_dif_ct2'], '.', label='Subcort GMV')
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_ct2'], '.', label='Cort GMV')
ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_pct_ra_dif_ct2'], '.', label='Cereb WMV')
ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of rac from ra')
ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_ra'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ra', scatter=False, ax=ax, color=cp[0])
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_ra'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ra', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_ra'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ra', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
fig.legend()

In [ ]:
global_df.loc[: ,global_df.columns[global_df.columns.str.contains('Total')]] /= 10000

In [ ]:
global_df.loc[: ,global_df.columns[global_df.columns.str.contains('Subcortical')]] /= 10000

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(10,5))
ax = axes[0,0]
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ra', scatter=False, ax=ax,label='RA')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1', scatter=False, ax=ax,label='RACT1')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1r5', scatter=False, ax=ax,label='RACT1-R5')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract2', scatter=False, ax=ax,label='RACT2')
ax.set_ylabel('Cortical GMV')
ax.set_xlabel('Age (Years)')

ax = axes[0,1]
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ra', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1r5', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract2', scatter=False, ax=ax)
ax.set_ylabel('Cerebral WMV')
ax.set_xlabel('Age (Years)')

ax = axes[0,2]
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ra', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1r5', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract2', scatter=False, ax=ax)
ax.set_ylabel('Subcortical GMV')
ax.set_xlabel('Age (Years)')

fig.legend(loc='upper right')
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(1)
sns.regplot(data=global_df, x='age_years', y='Total cortical gra matter volume_ra', scatter=False, ax=ax,label='RA')
sns.regplot(data=global_df, x='age_years', y='Total cortical gra matter volume_ract1', scatter=False, ax=ax,label='RACT1')
sns.regplot(data=global_df, x='age_years', y='Total cortical gra matter volume_ract1r3', scatter=False, ax=ax,label='RACT1-R3')
sns.regplot(data=global_df, x='age_years', y='Total cortical gra matter volume_ract1r5', scatter=False, ax=ax,label='RACT1-R5')
sns.regplot(data=global_df, x='age_years', y='Total cortical gra matter volume_ract2', scatter=False, ax=ax,label='RACT2')



fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ra', scatter=False, ax=ax, label="RA")
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1', scatter=False, ax=ax, label="RACT1")
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1r3', scatter=False, ax=ax, label="RACT1-R3")
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1r5', scatter=False, ax=ax, label="RACT1-R5")
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract2', scatter=False, ax=ax, label="RACT2")



fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_ract1'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1', scatter=False, ax=ax, color=cp[0])
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_ract1'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_ract1'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_ract1r5'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1r5', scatter=False, ax=ax, color=cp[0])
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_ract1r5'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1r5', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_ract1r5'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1r5', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
fig.legend()

In [ ]:
fig, ax = plt.subplots(1)
ax.plot(global_df.age_years, global_df['Subcortical gray matter volume_pct_ra_dif_ct1'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_pct_ra_dif_ct1', scatter=False, ax=ax, color=cp[0])
ax.plot(global_df.age_years, global_df['Total cortical gray matter volume_pct_ra_dif_ct1'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_pct_ra_dif_ct1', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df['Total cerebral white matter volume_pct_ra_dif_ct1'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_pct_ra_dif_ct1', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
fig.legend()

In [ ]:
# T2
source = 'not2'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all noT2')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all noT2')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all noT2')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all noT2 from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = 'ct1r3'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = 'ct1r5'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = 'ct1'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Cort GMV, $\beta=0.40, t=15.78, p=3.15e^{-50}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Subcort GMV, $\beta=0.014, t=0.36, p=0.72$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label=r'Cereb WMV, $\beta=-0.17, t=-6.24, p=6.43e^{-10}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.suptitle("recon-all vs recon-all clinical with high-res T1w input")
fig.tight_layout()


In [ ]:
source = 'ct1r5'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
source = 'ct2'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label='Cort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label='Cereb WMV')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label='Subcort GMV')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])

ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.tight_layout()

In [ ]:
metric = 'Total cortical gray matter volume'
g = sns.scatterplot(data=global_df.loc[global_df[f'{metric}_ract1r5'].notnull()], x=f'{metric}_ract1', y=f'{metric}_ract1r5', hue='age_years')
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all from highres T1')
g.set_ylabel('recon-all clinical from highres T2')
g.set_title(f'recon-all clinical estimated\n{metric}')

In [ ]:
metric = 'Total cortical gray matter volume'
g = sns.scatterplot(data=global_df, x=f'{metric}_ra', y=f'{metric}_ract2', hue='age_years')
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all from highres T1')
g.set_ylabel('recon-all clinical from highres T2')
g.set_title(f'recon-all clinical estimated\n{metric}')

In [ ]:
metric = 'Total cerebral white matter volume'
g = sns.scatterplot(data=global_df, x=f'{metric}_ract1', y=f'{metric}_ract2', hue='age_years')
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([100000, 900000], [100000, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('Estimated from highres T1')
g.set_ylabel('Estimated from highres T2')
g.set_title(f'recon-all clinical estimated\n{metric}')

In [ ]:
metric = 'Subcortical gray matter volume'
g = sns.scatterplot(data=global_df, x=f'{metric}_ract1', y=f'{metric}_ract2', hue='age_years')
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_ylabel('Estimated from highres T1')
g.set_xlabel('Estimated from highres T2')
g.set_title(f'recon-all clinical estimated\n{metric}')

In [ ]:
metric = 'Total gray matter volume'
g = sns.scatterplot(data=global_df, x=f'{metric}_ract1', y=f'{metric}_ract2', hue='age_years')
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_ylabel('Estimated from highres T1')
g.set_xlabel('Estimated from highres T2')
g.set_title(f'recon-all clinical estimated\n{metric}')

In [ ]:
global_df.age_years.min(), global_df.age_years.max()

In [ ]:
global_df['Subcort_GMV_pct_dif_t1'] = global_df['Subcortical gray matter volume_pct_ra_dif_ct1']
global_df['Cort_GMV_pct_dif_t1'] = global_df['Total cortical gray matter volume_pct_ra_dif_ct1']
global_df['Cereb_WMV_pct_dif_t1'] = global_df['Total cerebral white matter volume_pct_ra_dif_ct1']

global_df['Subcort_GMV_pct_dif_t1r5'] = global_df['Subcortical gray matter volume_pct_ra_dif_ct1r5']
global_df['Cort_GMV_pct_dif_t1r5'] = global_df['Total cortical gray matter volume_pct_ra_dif_ct1r5']
global_df['Cereb_WMV_pct_dif_t1r5'] = global_df['Total cerebral white matter volume_pct_ra_dif_ct1r5']

global_df['Subcort_GMV_pct_dif_t2'] = global_df['Subcortical gray matter volume_pct_ra_dif_ct2']
global_df['Cort_GMV_pct_dif_t2'] = global_df['Total cortical gray matter volume_pct_ra_dif_ct2']
global_df['Cereb_WMV_pct_dif_t2'] = global_df['Total cerebral white matter volume_pct_ra_dif_ct2']

global_df['Subcort_GMV_pct_dif_t1vt2'] = global_df['Subcortical gray matter volume_pct_ct1_dif_ct2']
global_df['Cort_GMV_pct_dif_t1vt2'] = global_df['Total cortical gray matter volume_pct_ct1_dif_ct2']
global_df['Cereb_WMV_pct_dif_t1vt2'] = global_df['Total cerebral white matter volume_pct_ct1_dif_ct2']



In [ ]:
scgmv_df = global_df.loc[:, ['Subcort_GMV_pct_dif', 'age_years']].rename(columns={'Subcort_GMV_pct_dif':'pct_dif'})
scgmv_df['metric'] = 'Subcort_GMV'
cgmv_df = global_df.loc[:, ['Cort_GMV_pct_dif', 'age_years']].rename(columns={'Cort_GMV_pct_dif':'pct_dif'})
cgmv_df['metric'] = 'Cort_GMV'
cwmv_df = global_df.loc[:, ['Cereb_WMV_pct_dif', 'age_years']].rename(columns={'Cereb_WMV_pct_dif':'pct_dif'})
cwmv_df['metric'] = 'Cereb_WMV'
pct_dif_df = pd.concat([cgmv_df, cwmv_df, scgmv_df])

In [ ]:
metric_cols

In [ ]:
difs = ['t1', 't1r5', 't2']

In [ ]:
global_df_nospace = global_df.copy()
global_df_nospace.columns = global_df_nospace.columns.str.replace(' ', '_')
cfg.metric_cols_nospace = [mc.replace(' ', '_') for mc in cfg.metric_cols]
global_df_nospace['years_9'] = global_df_nospace.age_years - 9

In [ ]:
global_resses = []
for dif in difs:
    for mc in cfg.metric_cols_nospace:
        mdl = smf.ols(f'{mc}_pct_ra_dif_c{dif} ~ age_years', data=global_df_nospace)
        mdl_fitted = mdl.fit()
        mdl_fitted.summary()
        res = mdl_fitted.summary2().tables[1].reset_index().rename(columns={'index':'vn'})
        res['metric']=mc
        res['dif'] = dif
        global_resses.append(res)
global_resses=pd.concat(global_resses)

In [ ]:
global_resses['RAC Input'] = global_resses.dif.replace({'t1':'RACT1', 't1r5' : 'RACT1-R5', 't2': 'RACT2'})

In [ ]:
tab_for_fig = global_resses.copy()

In [ ]:
tab_for_fig = tab_for_fig.rename(columns={'vn': 'Term', 'Coef.': 'Coef', 'P>|t|':'p',})

In [ ]:
tab_for_fig['Measure'] = tab_for_fig.metric.replace({'Subcortical_gray_matter_volume': 'Subcort GMV',
                                                     'Total_cortical_gray_matter_volume': 'Cort GMV',
                                                     'Total_cerebral_white_matter_volume': 'Cereb WMV',
                                                     'Brain_Segmentation_Volume_Without_Ventricles': 'Brain V'})

In [ ]:
tab_measures = ['Subcort GMV', 'Cort GMV', 'Cereb WMV']
tab_for_fig = tab_for_fig.loc[tab_for_fig.Measure.isin(tab_measures), ['Measure', 'RAC Input', 'Term', 'Coef', 't', 'p']].copy()

In [ ]:
tab_for_fig['Coef'] = tab_for_fig.Coef.apply(lambda x: f'{x:0.2f}')
tab_for_fig['t'] = tab_for_fig.t.apply(lambda x: f'{x:0.2f}')
tab_for_fig['p'] = tab_for_fig.p.apply(lambda x: f'{x:0.2g}*' if x < 0.05 else f'{x:0.2g}')

In [ ]:
tab_for_fig.sort_values(['RAC Input', 'Measure', 'Term'])

In [ ]:
tab_for_fig = tab_for_fig.sort_values(['RAC Input', 'Measure', 'Term']).set_index(['RAC Input','Measure', 'Term']).unstack(0).swaplevel(axis=1).T.reset_index().sort_values(['RAC Input', 'level_1']).set_index(['RAC Input', 'level_1']).T.loc[tff_order]

In [ ]:
tab_for_fig.columns.names =['RAC Input', '']
tab_for_fig

In [ ]:
tab_for_fig.columns

In [ ]:
tff_rorder = [
    (   'Cort GMV', 'Intercept'),
    (   'Cort GMV', 'age_years'),
    (  'Cereb WMV', 'Intercept'),
    (  'Cereb WMV', 'age_years'),
    ('Subcort GMV', 'Intercept'),
    ('Subcort GMV', 'age_years')
]
tff_corder = [
    (   'RACT1', 'Coef'),
    (   'RACT1',    't'),
    (   'RACT1',    'p'),
    (   'RACT1-R5', 'Coef'),
    (   'RACT1-R5',    't'),
    (   'RACT1-R5',    'p'),
    (   'RACT2', 'Coef'),
    (   'RACT2',    't'),
    (   'RACT2',    'p'),
]
tab_for_fig.loc[tff_rorder, tff_corder]

In [ ]:
global_resses.query("vn == 'age_years' & dif == 't1'").sort_values(['metric', 'dif'])

In [ ]:
global_resses.query("vn == 'age_years' & dif == 't2'").sort_values(['metric', 'dif'])

In [ ]:
dir(g.legend_.legend_handles)

In [ ]:
fig, ax = plt.subplots(1)
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = g.get_ylim()
xlim = g.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1 ($10^5$ mm$^3$)')
ax.set_title(f'Cort GMV, $r^2={r2:0.2f}$')
fig.colorbar(cm.ScalarMappable(norm=norm, cmap=sns.color_palette('flare', as_cmap=True)), ax=ax)

In [ ]:
fig = plt.figure(figsize=(10,13), dpi=100, layout='constrained')
norm = colors.Normalize(vmin=9, vmax=18)
subfigs = fig.subfigures(2, 1, height_ratios=[3.1,1.1])
axes = subfigs[0].subplots(3,3)
# gs = GridSpec(21, 3,wspace=1, hspace=1, figure=fig)
# axes = np.array([
#     [fig.add_subplot(gs[:5, 0]), fig.add_subplot(gs[:5, 1]), fig.add_subplot(gs[:5, 2])],
#     [fig.add_subplot(gs[5:10, 0]), fig.add_subplot(gs[5:10, 1]), fig.add_subplot(gs[5:10, 2])],
#     [fig.add_subplot(gs[10:15, 0]), fig.add_subplot(gs[10:15, 1]), fig.add_subplot(gs[10:15, 2])],
#     [fig.add_subplot(gs[16:, 0]), fig.add_subplot(gs[16:, 1]), fig.add_subplot(gs[16:, 2])],
# ])
# legax = fig.add_subplot(gs[15,:])

source = 'ct1'

ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1 ($10^5$ mm$^3$)')
ax.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1 ($10^5$ mm$^3$)')
ax.set_title(f'Cereb WMV $r^2={r2:0.2f}$')
#handles = g.legend_.legend_handles

#legax.axis('off')
#legax.legend(handles=handles, ncols=6, loc='upper center', bbox_to_anchor=(0.5, 1), title='Age (years)')
# g.legend_.set_ncols(6)
# g.legend_.set_loc('upper center')
# g.legend_.set_bbox_to_anchor((0.5, 0))


ax = axes[2,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1 ($10^5$ mm$^3$)')
ax.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')




source = 'ct1r5'

ax = axes[0,1]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1-R5 ($10^5$ mm$^3$)')
ax.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[1,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1-R5 ($10^5$ mm$^3$)')
ax.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[2,1]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT1-R5 ($10^5$ mm$^3$)')
ax.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')



source = 'ct2'
ax = axes[0,2]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT2 ($10^5$ mm$^3$)')
ax.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[1,2]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT2 ($10^5$ mm$^3$)')
ax.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[2,2]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
ax.scatter(x=global_df[x], y=global_df[y], c=global_df.age_years, marker='.', cmap=sns.color_palette('flare', as_cmap=True), norm=norm)
ylim = ax.get_ylim()
xlim = ax.get_xlim()
ax.plot([0, 900000], [0, 900000], label='x=y', color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel('RA ($10^5$ mm$^3$)')
ax.set_ylabel('RACT2 ($10^5$ mm$^3$)')
ax.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

subfigs[0].colorbar(cm.ScalarMappable(norm=norm, cmap=sns.color_palette('flare', as_cmap=True)), label='Age (years)', ax=axes[2,:], location='bottom', shrink=0.4)

axes = subfigs[1].subplots(1,3)
source='ct1'
ax= axes[0]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cort GMV')#, $\beta=0.40, t=15.78, p=3.15e^{-50}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Subcort GMV')#, $\beta=0.014, t=0.36, p=0.72$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cereb WMV')#, $\beta=-0.17, t=-6.24, p=6.43e^{-10}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Diff. RACT1 from RA')
ax.set_ylim((-20, 20))
ax.set_title('RACT1')

#ax.legend()

souce='ct1r5'
ax= axes[1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cort GMV')#, $\beta=0.56, t=18.89, p=3.15e^{-68}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Subcort GMV')#, $\beta=0.09, t=2.21, p=0.027$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cereb WMV')#, $\beta=-0.15, t=-4.98, p=7.39e^{-7}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Diff. RACT1-R5 from RA')
ax.set_title('RACT1-R5')

ax.set_ylim((-20, 20))
#ax.legend()

source='ct2'
ax= axes[2]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cort GMV')#, $\beta=0.40, t=12.58, p=9.35e^{-34}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Subcort GMV')#, $\beta=-0.02, t=-0.45, p=0.65$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', markersize=2, label=r'Cereb WMV')#, $\beta=0.22, t=5.64, p=2.16e^{-8}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Diff. RACT2 from RA')
ax.set_ylim((-20, 20))
ax.set_title('RACT2')
tmpleg = ax.legend()
handles = tmpleg.legend_handles
tmpleg.set_visible(False)
subfigs[1].legend(handles=handles, loc='outside lower center', ncols=3, title='Measure', markerscale=4)
#fig.suptitle("recon-all vs recon-all clinical with high-res T1w input")

#fig.tight_layout()

plt.show()

In [ ]:
source = 'ct2'
fig, axes = plt.subplots(4,3, figsize=(10,40), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Cort GMV, $\beta=0.40, t=12.58, p=9.35e^{-34}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Subcort GMV, $\beta=-0.02, t=-0.45, p=0.65$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])

ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label=r'Cereb WMV, $\beta=0.22, t=5.64, p=2.16e^{-8}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.suptitle("recon-all vs recon-all clinical with high-res T1w input")

fig.tight_layout()

In [ ]:
source = 'ct1r5'
fig, axes = plt.subplots(2,2, figsize=(10,10), dpi=100)
ax = axes[0,0]
metric = 'Total cortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cort GMV, $r^2={r2:0.2f}$')

ax = axes[0,1]
metric = 'Total cerebral white matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Cereb WMV $r^2={r2:0.2f}$')

ax = axes[1,0]
metric = 'Subcortical gray matter volume'
x = f'{metric}_ra'
y = f'{metric}_ra{source}'
r2 = np.corrcoef(global_df.loc[global_df[y].notnull(), x], global_df.loc[global_df[y].notnull(), y])[0,1] ** 2
g = sns.scatterplot(data=global_df, x=x, y=y, hue='age_years', marker='o', ax=ax)
ylim = g.get_ylim()
xlim = g.get_xlim()
g.plot([0, 900000], [0, 900000], label='x=y', color='black')
g.set_xlim(xlim)
g.set_ylim(ylim)
g.set_xlabel('recon-all')
g.set_ylabel('recon-all clinical')
g.set_title(f'Subcort GMV, $r^2={r2:0.2f}$')

ax= axes[1,1]

ax.plot(global_df.age_years, global_df[f'Total cortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Cort GMV, $\beta=0.56, t=18.89, p=3.15e^{-68}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[0])

ax.plot(global_df.age_years, global_df[f'Subcortical gray matter volume_pct_ra_dif_{source}'], '.', label=r'Subcort GMV, $\beta=0.09, t=2.21, p=0.027$')
sns.regplot(data=global_df, x='age_years', y=f'Subcortical gray matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[1])
ax.plot(global_df.age_years, global_df[f'Total cerebral white matter volume_pct_ra_dif_{source}'], '.', label=r'Cereb WMV, $\beta=-0.15, t=-4.98, p=7.39e^{-7}$')
sns.regplot(data=global_df, x='age_years', y=f'Total cerebral white matter volume_pct_ra_dif_{source}', scatter=False, ax=ax, color=cp[2])


ax.set_xlabel('Age (years)')
ax.set_ylabel('% Difference of recon-all clinical from recon-all')
ax.set_ylim((-20, 20))
ax.legend()
fig.suptitle("recon-all vs recon-all clinical with 1x1x5mm T1w input")

fig.tight_layout()

In [ ]:
scgmv = smf.ols('Subcort_GMV_pct_dif_t1 ~ age_years', data=global_df)
scgmv_fitted = scgmv.fit()
scgmv_fitted.summary()

In [ ]:
cgmv = smf.ols('Cort_GMV_pct_dif_t1 ~ age_years', data=global_df)
cgmv_fitted = cgmv.fit()
cgmv_fitted.summary()

In [ ]:
cgmv_fitted.pvalues

In [ ]:
cwmv = smf.ols('Cereb_WMV_pct_dif_t1 ~ age_years', data=global_df)
cwmv_fitted = cwmv.fit()
cwmv_fitted.summary()

In [ ]:
cwmv_fitted.pvalues

In [ ]:
cwmv = smf.ols('Cereb_WMV_pct_dif_t2 ~ age_years', data=global_df)
cwmv_fitted = cwmv.fit()
cwmv_fitted.summary()

In [ ]:
cwmv = smf.ols('Cereb_WMV_pct_dif_t1vt2 ~ age_years', data=global_df)
cwmv_fitted = cwmv.fit()
cwmv_fitted.summary()

# assess region wise gray volume

In [ ]:
atlas='Desikan2006'
gv_df = pd.read_csv(f'../data/derivatives/fs_stats/recon-all/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df.columns = gv_df.columns.str.replace(' ', '_').str.replace('-','_')
cfg.metric_cols = gv_df.columns[2:]

gv_df_ct1 = pd.read_csv(f'../data/derivatives/fs_stats/recon-all_clinical_t1/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df_ct1.columns = gv_df_ct1.columns.str.replace(' ', '_').str.replace('-','_')

gv_df_ct1r5 = pd.read_csv(f'../data/derivatives/fs_stats/recon-all_clinical_t1_resample-5/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df_ct1r5.columns = gv_df_ct1r5.columns.str.replace(' ', '_').str.replace('-','_')
ct1r5_measures = [mm + '_ract1r5' for mm in gv_df_ct1r5.columns[2:]]
gv_df_ct1r5.columns = ['subject', 'session',] + ct1r5_measures

gv_df_ct1r3 = pd.read_csv(f'../data/derivatives/fs_stats/recon-all_clinical_t1_resample-3/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df_ct1r3.columns = gv_df_ct1r3.columns.str.replace(' ', '_').str.replace('-','_')
ct1r3_measures = [mm + '_ract1r3' for mm in gv_df_ct1r3.columns[2:]]
gv_df_ct1r3.columns = ['subject', 'session',] + ct1r3_measures

gv_df_ct2 = pd.read_csv(f'../data/derivatives/fs_stats/recon-all_clinical_t2/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df_ct2.columns = gv_df_ct2.columns.str.replace(' ', '_').str.replace('-','_')
ct2_measures = [mm + '_ract2' for mm in gv_df_ct2.columns[2:]]
gv_df_ct2.columns = ['subject', 'session',] + ct2_measures

gv_df_not2 = pd.read_csv(f'../data/derivatives/fs_stats/recon-all_not2/atlas-{atlas}_GrayVol.tsv', sep='\t').drop('temporalpole_rh', axis=1, errors='ignore')
gv_df_not2.columns = gv_df_not2.columns.str.replace(' ', '_').str.replace('-','_')
not2_measures = [mm + '_ranot2' for mm in gv_df_not2.columns[2:]]
gv_df_not2.columns = ['subject', 'session',] + not2_measures




In [ ]:
import templateflow
import nilearn.plotting as nlp
import nibabel as nb

In [ ]:
gv_df = gv_df.merge(gv_df_ct1, how='inner', on=['subject', 'session'], suffixes=['_ra', '_ract1'])
gv_df = gv_df.merge(gv_df_ct2, how='inner', on=['subject', 'session'])
gv_df = gv_df.merge(gv_df_not2, how='inner', on=['subject', 'session'])
gv_df = gv_df.merge(gv_df_ct1r3, how='left', on=['subject', 'session'])
gv_df = gv_df.merge(gv_df_ct1r5, how='left', on=['subject', 'session'])


ses_info['subject'] = ses_info.participant_id.str.split('-').str[1]
ses_info['session'] = ses_info.session_id.str.split('-').str[1]
ses_info['age_years'] = ses_info.age / 12
pml = len(gv_df)
gv_df = gv_df.merge(ses_info.loc[:, ['subject', 'session', 'age_years', 'site_id']], how='left', on=['subject', 'session'])
assert pml == len(gv_df)

gv_df = gv_df.merge(fastages.loc[:, ['subject', 'session', 'age_years']], how='left', on=['subject', 'session'], suffixes=['', '_fa'])
gv_df['age_years'] = gv_df.age_years.fillna(gv_df.age_years_fa)
assert len(gv_df.loc[gv_df.age_years.isnull()]) == 0

gv_df.columns = gv_df.columns.str.replace(' ', '_').str.replace('-','_')


In [ ]:
dif_cols_ct1 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1'
    dif_cols_ct1.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()


dif_cols_ct2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct2'
    dif_cols_ct2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract2'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()

dif_cols_not2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_not2'
    dif_cols_not2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ranot2'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()

dif_cols_ct1r3 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1r3'
    dif_cols_ct2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1r3'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()

dif_cols_ct1r5 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ra_dif_ct1r5'
    dif_cols_ct2.append(dif_col)
    ra_col = f'{mc}_ra'
    rac_col = f'{mc}_ract1r5'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()


dif_cols_ct1ct2 = []
for mc in cfg.metric_cols:
    dif_col = f'{mc}_pct_ct1_dif_ct2'
    dif_cols_ct2.append(dif_col)
    ra_col = f'{mc}_ract1'
    rac_col = f'{mc}_ract2'
    gv_df[dif_col] = ((gv_df[rac_col] - gv_df[ra_col]) / gv_df[ra_col] ) * 100
    gv_df = gv_df.copy()



In [ ]:
difs = ['_pct_ra_dif_ct1','_pct_ra_dif_ct1r3', '_pct_ra_dif_ct1r5', '_pct_ra_dif_ct2', '_pct_ct1_dif_ct2']

In [ ]:
gv_df['centered_age'] = gv_df.age_years - gv_df.age_years.mean()

In [ ]:
resses = []
for dif in difs:
    for mc in cfg.metric_cols:
        mdl = smf.ols(f'{mc}{dif} ~ age_years', data=gv_df)
        mdl_fitted = mdl.fit()
        mdl_fitted.summary()
        res = mdl_fitted.summary2().tables[1].reset_index().rename(columns={'index':'vn'})
        res['metric']=mc
        res['dif'] = dif
        resses.append(res)
resses=pd.concat(resses)

In [ ]:
sig, q, _, _ = multipletests(resses.query('vn == "age_years"')['P>|t|'], method='fdr_by')
resses.loc[resses.vn == 'age_years', 'q'] = q
resses.loc[resses.vn == 'age_years', 'fdr_sig'] = sig
sig, q, _, _ = multipletests(resses.query('vn == "Intercept"')['P>|t|'], method='fdr_by')
resses.loc[resses.vn == 'Intercept', 'q'] = q
resses.loc[resses.vn == 'Intercept', 'fdr_sig'] = sig

In [ ]:
templateflow.api.get('fsaverage', atlas='Desikan2006')

In [ ]:

templateflow.api.get('fsaverage')#, density='3k')

In [ ]:
desikan_lh = nb.load('/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_atlas-Desikan2006_seg-aparc_desc-curated_dseg.label.gii')
desikan_lh_data = desikan_lh.agg_data()

desikan_lut = pd.read_csv('/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_atlas-Desikan2006_dseg.tsv', sep='\t')
desikan_lh_lut = desikan_lut.query('hemi == "L"').reset_index(names='val')
desikan_lh_lut['name_h'] = desikan_lh_lut.name + '_lh'

desikan_rh = nb.load('/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_atlas-Desikan2006_seg-aparc_desc-curated_dseg.label.gii')
desikan_rh_data = desikan_rh.agg_data()

desikan_rh_lut = desikan_lut.query('hemi == "R"').reset_index(drop=True).reset_index(names='val')
desikan_rh_lut['name_h'] = desikan_rh_lut.name + '_rh'

In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in to_plot_lh_df.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['Coef.']


to_plot_rh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("rh")')
to_plot_rh_df = to_plot_rh_df.merge(desikan_rh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_rh_vals =  np.zeros_like(desikan_rh_data, dtype=float)
for ix, row in to_plot_rh_df.iterrows():
    surf_rh_vals[desikan_rh_data == row.val] = row['Coef.']
    

In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in to_plot_lh_df.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['Coef.']


to_plot_rh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("rh")')
to_plot_rh_df = to_plot_rh_df.merge(desikan_rh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_rh_vals =  np.zeros_like(desikan_rh_data, dtype=float)
for ix, row in to_plot_rh_df.iterrows():
    surf_rh_vals[desikan_rh_data == row.val] = row['Coef.']
    
fig, axes = plt.subplots(1,4, subplot_kw={'projection': '3d'}, figsize=(10,5), dpi=100)
g = nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[0],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[1],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[2],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[3],
    vmax=1,
    vmin=-1,
    colorbar=True
)


In [ ]:
fig,ax = plt.subplots()
norm = colors.Normalize(vmin=-1, vmax=1)

fig.colorbar(cm.ScalarMappable(norm=norm, cmap=sns.color_palette('RdBu_r', as_cmap=True)), orientation='horizontal',
                    label=r'$\beta_{age}$ from model: $pct\_err\_GMV \sim age + 1$', ax=ax, ticks=[-1, -0.5, 0, 0.5, 1], shrink=0.7)


In [ ]:
fig,ax = plt.subplots()
norm = colors.Normalize(vmin=0, vmax=15)

fig.colorbar(cm.ScalarMappable(norm=norm, cmap=sns.color_palette('Oranges', as_cmap=True)), orientation='horizontal',
                    label='Age of peak GMV (years)', ax=ax, shrink=0.7, ticks=[0,5, 10, 15])


In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in to_plot_lh_df.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['Coef.']


to_plot_rh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1" & metric.str.contains("rh")')
to_plot_rh_df = to_plot_rh_df.merge(desikan_rh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_rh_vals =  np.zeros_like(desikan_rh_data, dtype=float)
for ix, row in to_plot_rh_df.iterrows():
    surf_rh_vals[desikan_rh_data == row.val] = row['Coef.']
    
fig, axes = plt.subplots(1,4, subplot_kw={'projection': '3d'}, figsize=(10,5), dpi=100)
g = nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[0],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[1],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[2],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[3],
    vmax=1,
    vmin=-1,
    colorbar=False
)


In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in to_plot_lh_df.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['Coef.']


to_plot_rh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5" & metric.str.contains("rh")')
to_plot_rh_df = to_plot_rh_df.merge(desikan_rh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_rh_vals =  np.zeros_like(desikan_rh_data, dtype=float)
for ix, row in to_plot_rh_df.iterrows():
    surf_rh_vals[desikan_rh_data == row.val] = row['Coef.']
    
fig, axes = plt.subplots(1,4, subplot_kw={'projection': '3d'}, figsize=(10,5), dpi=100)
g = nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[0],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[1],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='lateral',
    axes=axes[2],
    colorbar=False,
    vmax=1,
    vmin=-1
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_164k_inflated.surf.gii',
    stat_map=surf_rh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-R_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='RdBu_r',
    view='medial',
    axes=axes[3],
    vmax=1,
    vmin=-1,
    colorbar=True
)


In [ ]:
regional_peaks = pd.read_csv("../data/lifespan_supplements/Table2_Milestones/Table_2_2.csv")
regional_peaks = regional_peaks.iloc[:, [0,1]].copy()
regional_peaks.columns=['region', 'volume_peak']
regional_peaks = regional_peaks.merge(desikan_lh_lut.loc[:, ['val', 'name']], how='left', left_on='region', right_on='name')

In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in regional_peaks.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['volume_peak']

    
fig, axes = plt.subplots(1,2, subplot_kw={'projection': '3d'}, figsize=(10,5), dpi=100)
g = nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='Oranges_r',
    view='lateral',
    axes=axes[0],
    colorbar=False,
    vmax=15,
    vmin=0
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='Oranges_r',
    view='medial',
    axes=axes[1],
    colorbar=False,
    vmax=15,
    vmin=0
)

In [ ]:
to_plot_lh_df = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5" & metric.str.contains("lh")')
to_plot_lh_df = to_plot_lh_df.merge(desikan_lh_lut.loc[:,['val', 'name_h']] , how='left', left_on='metric', right_on='name_h')
surf_lh_vals =  np.zeros_like(desikan_lh_data, dtype=float)
for ix, row in regional_peaks.iterrows():
    surf_lh_vals[desikan_lh_data == row.val] = row['volume_peak']

    
fig, axes = plt.subplots(1,2, subplot_kw={'projection': '3d'}, figsize=(10,5), dpi=100)
g = nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='Oranges',
    view='lateral',
    axes=axes[0],
    colorbar=False,
    vmax=15,
    vmin=0
)
nlp.plot_surf_stat_map(
    '~/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_164k_inflated.surf.gii',
    stat_map=surf_lh_vals,
    bg_map='/home/nielsond/.cache/templateflow/tpl-fsaverage/tpl-fsaverage_hemi-L_den-164k_sulc.shape.gii', 
    bg_on_data=True,
    cmap='Oranges',
    view='medial',
    axes=axes[1],
    colorbar=True,
    vmax=15,
    vmin=0
)

In [ ]:
to_corr = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1"').copy()
to_corr['region'] = to_corr.metric.str.split('_').str[0]
to_corr['hemisphere'] = to_corr.metric.str.split('_').str[1]

to_corr = to_corr.merge(regional_peaks, how='left', on='region')

In [ ]:
to_corr.loc[to_corr.hemisphere == 'lh', ['region', 'hemisphere']].merge(to_corr.loc[to_corr.hemisphere == 'rh', ['region', 'hemisphere']], how='outer', on='region', indicator=True)

In [ ]:
stats.pearsonr(to_corr.query('hemisphere == "lh"')['Coef.'], to_corr.query('hemisphere == "lh"')['volume_peak'])

In [ ]:
(-0.5852530548111721)**2

In [ ]:
stats.pearsonr(to_corr.query('hemisphere == "rh"')['Coef.'], to_corr.query('hemisphere == "rh"')['volume_peak'])

In [ ]:
sns.lmplot(data=to_corr, x='volume_peak', y='Coef.', hue='hemisphere')

In [ ]:
to_corr = resses.query('vn == "age_years" & dif == "_pct_ra_dif_ct1r5"').copy()
to_corr['region'] = to_corr.metric.str.split('_').str[0]
to_corr['hemisphere'] = to_corr.metric.str.split('_').str[1]

to_corr = to_corr.merge(regional_peaks, how='left', on='region')
print(stats.pearsonr(to_corr.query('hemisphere == "lh"')['Coef.'], to_corr.query('hemisphere == "lh"')['volume_peak']))
print(stats.pearsonr(to_corr.query('hemisphere == "rh"')['Coef.'], to_corr.query('hemisphere == "rh"')['volume_peak']))

In [ ]:
sns.lmplot(data=to_corr, x='volume_peak', y='Coef.', hue='hemisphere')

In [ ]:
to_corr.merge(regional_peaks, how='left', on='region')

In [ ]:
resses.query('dif == "_pct_ra_dif_ct1" & vn == "Intercept"').sort_values('t')

In [ ]:
resses.query('dif == "_pct_ra_dif_ct1" & vn == "centered_age"').sort_values('t').to_feather(f'../data/derivatives/fs_stats/var-age_atlas-{atlas}.feather')

In [ ]:
resses.query('dif == "_pct_ct1_dif_ct2" & vn == "centered_age"').sort_values('t')

# How badly does the error mess up models of development

In [ ]:
from sklearn import linear_model

In [ ]:
fix_df = gv_df.reset_index(drop=True)
fix_df['centered_age_squared'] = fix_df.centered_age ** 2
fix_df['age_years_squared'] = fix_df.age_years ** 2

fix_df['centered_age_cubed'] = fix_df.centered_age ** 3

tr_df = fix_df.sample(frac=0.5)
te_df = fix_df.loc[~fix_df.index.isin(tr_df.index)].copy()

In [ ]:
tr_df.age_years.describe()

In [ ]:
siteresses = []
for dif in [difs[0]]:
    for mc in cfg.metric_cols:
        mdl = smf.ols(f'{mc}{dif} ~ centered_age*site_id', data=fix_df)
        mdl_fitted = mdl.fit()
        mdl_fitted.summary()
        res = mdl_fitted.summary2().tables[1].reset_index().rename(columns={'index':'vn'})
        res['metric']=mc
        res['dif'] = dif
        siteresses.append(res)
siteresses=pd.concat(siteresses)

sig, q, _, _ = multipletests(siteresses.query('vn == "centered_age"')['P>|t|'], method='fdr_by')
siteresses.loc[siteresses.vn == 'centered_age', 'q'] = q
siteresses.loc[siteresses.vn == 'centered_age', 'fdr_sig'] = sig

sig, q, _, _ = multipletests(siteresses.query('vn == "Intercept"')['P>|t|'], method='fdr_by')
siteresses.loc[siteresses.vn == 'Intercept', 'q'] = q
siteresses.loc[siteresses.vn == 'Intercept', 'fdr_sig'] = sig

sig, q, _, _ = multipletests(siteresses.loc[siteresses.vn.str.contains('site'), 'P>|t|'], method='fdr_by')
siteresses.loc[siteresses.vn.str.contains('site'), 'q'] = q
siteresses.loc[siteresses.vn.str.contains('site'), 'fdr_sig'] = sig

# site effects mostly don't surive multiple comparisons
siteresses.loc[siteresses.vn.str.contains('site') & siteresses.fdr_sig, :]

In [ ]:
        z_df = fix_df.copy()


In [ ]:
fix_resses = []
sources=['ra', 'ract1', 'ract1r3', 'ract1r5', 'ract2']
for source in sources:
    tmp = []
    for mc in cfg.metric_cols:
        z_df[f'{mc}_{source}'] = (z_df[f'{mc}_{source}'] - z_df[f'{mc}_{source}'].mean()) / z_df[f'{mc}_{source}'].std()
        trmdl = smf.rlm(f'{mc}_{source} ~ age_years + age_years_squared', data=z_df)
        trmdl_fitted = trmdl.fit()
        trres = pd.DataFrame(trmdl_fitted.summary2().tables[1].to_dict())
        trres['source'] = source
        trres['region'] = mc
        trres['split'] = 'full'
        trres['sig'] = trres['P>|z|'] < 0.05
        trres['sign'] = np.sign(trres['Coef.'])
        trres = trres.reset_index(names=['term'])
        tmp.append(trres)
    tmp = pd.concat(tmp)
    sig, q, _, _ = multipletests(tmp.query('term == "Intercept"')['P>|z|'], method='fdr_by')
    tmp.loc[tmp.term == 'Intercept', 'q'] = q
    tmp.loc[tmp.term == 'Intercept', 'fdr_sig'] = sig
    
    sig, q, _, _ = multipletests(tmp.query('term == "age_years"')['P>|z|'], method='fdr_by')
    tmp.loc[tmp.term == 'age_years', 'q'] = q
    tmp.loc[tmp.term == 'age_years', 'fdr_sig'] = sig
    
    sig, q, _, _ = multipletests(tmp.query('term == "age_years_squared"')['P>|z|'], method='fdr_by')
    tmp.loc[tmp.term == 'age_years_squared', 'q'] = q
    tmp.loc[tmp.term == 'age_years_squared', 'fdr_sig'] = sig
    fix_resses.append(tmp)
fix_resses = pd.concat(fix_resses)
fix_resses = fix_resses.rename(columns={'Coef.':'coef', 'P>|z|':'p'})

In [ ]:
sig, q, _, _ = multipletests(fix_resses.query('term == "Intercept"')['p'], method='fdr_by')
fix_resses.loc[fix_resses.term == 'Intercept', 'q'] = q
fix_resses.loc[fix_resses.term == 'Intercept', 'fdr_sig'] = sig

sig, q, _, _ = multipletests(fix_resses.query('term == "age_years"')['p'], method='fdr_by')
fix_resses.loc[fix_resses.term == 'age_years', 'q'] = q
fix_resses.loc[fix_resses.term == 'age_years', 'fdr_sig'] = sig

sig, q, _, _ = multipletests(fix_resses.query('term == "age_years_squared"')['p'], method='fdr_by')
fix_resses.loc[fix_resses.term == 'age_years_squared', 'q'] = q
fix_resses.loc[fix_resses.term == 'age_years_squared', 'fdr_sig'] = sig

In [ ]:
fix_resses = fix_resses.rename(columns={'Coef.':'coef'})

In [ ]:
stats.pearsonr(fix_resses.query('term == "age_years" & source=="ra"').sign, fix_resses.query('term == "age_years" & source=="ract1"').sign)

In [ ]:
(fix_resses.query('term == "age_years" & source=="ra"').sign == fix_resses.query('term == "age_years" & source=="ract1"').sign).mean()

In [ ]:
(fix_resses.query('term == "age_years" & source=="ra"').sign == fix_resses.query('term == "age_years" & source=="ract1r5"').sign).mean()

In [ ]:
sns.regplot(x=fix_resses.query('term == "age_years" & source=="ra"').coef, y=fix_resses.query('term == "age_years" & source=="ract1"').coef)

In [ ]:
fix_resses.pivot(

In [ ]:
age_ps = fix_resses.loc[fix_resses.term == 'age_years', ['source', 'region', 'p']].pivot(columns=['source'], index='region')
age_ps.columns = [cc[1] for cc in age_ps.columns]

In [ ]:
age_ps

In [ ]:
sns.regplot(z_df, x='age_years', y='rostralmiddlefrontal_rh_ra')

In [ ]:
age_coefs = fix_resses.loc[fix_resses.term == 'age_years', ['source', 'region', 'coef']].pivot(columns=['source'], index='region')
age_coefs.columns = [cc[1] for cc in age_coefs.columns]

age2_coefs = fix_resses.loc[fix_resses.term == 'age_years_squared', ['source', 'region', 'coef']].pivot(columns=['source'], index='region')
age2_coefs.columns = [cc[1] for cc in age2_coefs.columns]

In [ ]:
fig, axes = plt.subplots(2,3, figsize=(10,6))
ax = axes[0,0]
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ra', scatter=False, ax=ax,label='RA')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1', scatter=False, ax=ax,label='RACT1')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract1r5', scatter=False, ax=ax,label='RACT1-R5')
sns.regplot(data=global_df, x='age_years', y='Total cortical gray matter volume_ract2', scatter=False, ax=ax,label='RACT2')
ax.set_ylabel('Cort GMV ($10^5$ mm$^3$)')
ax.set_xlabel('Age (Years)')

ax = axes[0,1]
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ra', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract1r5', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Total cerebral white matter volume_ract2', scatter=False, ax=ax)
ax.set_ylabel('Cereb WMV ($10^5$ mm$^3$)')
ax.set_xlabel('Age (Years)')

ax = axes[0,2]
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ra', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract1r5', scatter=False, ax=ax)
sns.regplot(data=global_df, x='age_years', y='Subcortical gray matter volume_ract2', scatter=False, ax=ax)
ax.set_ylabel('Subcort GMV ($10^5$ mm$^3$)')
ax.set_xlabel('Age (Years)')

ax = axes[1,0]
sns.regplot(data=age_coefs, x='ra', y='ract1', robust=True, ax=ax, color=cp[1])
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.plot([-10000,10000], [-10000, 10000], color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r'$\beta_{age}$ from RA')
ax.set_ylabel(r'$\beta_{age}$ from RACT1')
r2 = np.corrcoef(age_coefs.ra, age_coefs.ract1)[0,1]
ax.set_title(f'$r^2$ = {r2:0.2f}')

ax = axes[1,1]
sns.regplot(data=age_coefs, x='ra', y='ract1r5', robust=True, ax=ax, color=cp[2])
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.plot([-10000,10000], [-10000, 10000], color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r'$\beta_{age}$ from RA')
ax.set_ylabel(r'$\beta_{age}$ from RACT1-R5')
r2 = np.corrcoef(age_coefs.ra, age_coefs.ract1r5)[0,1]
ax.set_title(f'$r^2$ = {r2:0.2f}')

ax = axes[1,2]
sns.regplot(data=age_coefs, x='ra', y='ract2', robust=True, ax=ax, color=cp[3])
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.plot([-10000,10000], [-10000, 10000], color='black')
ax.set_xlim(xlim)
ax.set_ylim(ylim)
ax.set_xlabel(r'$\beta_{age}$ from RA')
ax.set_ylabel(r'$\beta_{age}$ from RACT2')
r2 = np.corrcoef(age_coefs.ra, age_coefs.ract2)[0,1]
ax.set_title(f'$r^2$ = {r2:0.2f}')

# ax = axes[2,0]
# sns.regplot(data=age2_coefs, x='ra', y='ract1', robust=True, ax=ax, color=cp[1])
# xlim = ax.get_xlim()
# ylim = ax.get_ylim()
# ax.plot([-10000,10000], [-10000, 10000], color='black')
# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# ax.set_xlabel(r'$\beta_{age^2}$ from RA')
# ax.set_ylabel(r'$\beta_{age^2}$ from RACT1')
# r2 = np.corrcoef(age2_coefs.ra, age2_coefs.ract1)[0,1]
# ax.set_title(f'$r^2$ = {r2:0.2f}')

# ax = axes[2,1]
# sns.regplot(data=age2_coefs, x='ra', y='ract1r5', robust=True, ax=ax, color=cp[2])
# xlim = ax.get_xlim()
# ylim = ax.get_ylim()
# ax.plot([-10000,10000], [-10000, 10000], color='black')
# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# ax.set_xlabel(r'$\beta_{age^2}$ from RA')
# ax.set_ylabel(r'$\beta_{age^2}$ from RACT1-R5')
# r2 = np.corrcoef(age2_coefs.ra, age2_coefs.ract1r5)[0,1]
# ax.set_title(f'$r^2$ = {r2:0.2f}')

# ax = axes[2,2]
# sns.regplot(data=age2_coefs, x='ra', y='ract2', robust=True, ax=ax, color=cp[3])
# xlim = ax.get_xlim()
# ylim = ax.get_ylim()
# ax.plot([-10000,10000], [-10000, 10000], color='black')
# ax.set_xlim(xlim)
# ax.set_ylim(ylim)
# ax.set_xlabel(r'$\beta_{age^2}$ from RA')
# ax.set_ylabel(r'$\beta_{age^2}$ from RACT2')
# r2 = np.corrcoef(age2_coefs.ra, age2_coefs.ract2)[0,1]
# ax.set_title(f'$r^2$ = {r2:0.2f}')

fig.legend(loc='center',bbox_to_anchor=[0.5, 0.53], ncols=4)

fig.tight_layout(h_pad=3)
fig.text(0,1, 'A', fontweight='bold')
fig.text(0, 0.5, 'B', fontweight='bold')

In [ ]:
stats.pearsonr(fix_resses.query('term == "age_years" & source=="ra"').coef, fix_resses.query('term == "age_years" & source=="ract1r5"').coef)

In [ ]:
stats.pearsonr(fix_resses.query('term == "age_years" & source=="ra"').coef, fix_resses.query('term == "age_years" & source=="ract2"').coef)

In [ ]:
tr_resses = []
te_resses = []
sources=['ra', 'ract1', 'ract1r3', 'ract1r5', 'ract2']
for source in sources:
    for mc in cfg.metric_cols:
        trmdl = smf.ols(f'{mc}_{source} ~ age_years + age_years_squared', data=tr_df)
        trmdl_fitted = trmdl.fit()
        temdl = smf.ols(f'{mc}_{source} ~ age_years + age_years_squared', data=te_df)
        temdl_fitted = temdl.fit()
        trres = pd.DataFrame(trmdl_fitted.summary2().tables[1].to_dict())
        trres['source'] = source
        trres['region'] = mc
        trres['split'] = 'train'
        trres['sig'] = trres['P>|t|'] < 0.05
        trres['sign'] = np.sign(trres['Coef.'])
        trres = trres.reset_index(names=['term'])
        tr_resses.append(trres)
        teres = pd.DataFrame(temdl_fitted.summary2().tables[1].to_dict())
        teres['source'] = source
        teres['region'] = mc
        teres['split'] = 'train'
        teres['sig'] = teres['P>|t|'] < 0.05
        teres['sign'] = np.sign(teres['Coef.'])
        teres = teres.reset_index(names=['term'])
        te_resses.append(teres)
tr_resses = pd.concat(tr_resses)
te_resses = pd.concat(te_resses)

In [ ]:
tr_resses.query("sig & term != 'Intercept'")

In [ ]:
te_resses.query("sig & term != 'Intercept'")

In [ ]:
mdl_resses = []
sources=['ract1', 'ract1r3', 'ract1r5', 'ract2']
for train_source in sources:
    for test_source in sources:
        if test_source != train_source:
            continue
        tmptedf = te_df.copy()
        for mc in cfg.metric_cols:
            mdl = smf.ols(f'{mc}_ra ~ centered_age*{mc}_{train_source} + centered_age_squared*{mc}_{train_source}', data=tr_df)
            mdl_fitted = mdl.fit()
            tmptedf[f'{mc}_{train_source}']=tmptedf[f'{mc}_{test_source}']
            pred = mdl_fitted.predict(tmptedf)
            error = pred - te_df[f'{mc}_ra']
            pct_error = (error / te_df[f'{mc}_ra']) * 100
            nomdl_error = tmptedf[f'{mc}_{train_source}'] - te_df[f'{mc}_ra'] 
            pct_nomdl_error = (nomdl_error / te_df[f'{mc}_ra']) * 100
            null_error = tr_df[f'{mc}_ra'].mean() - te_df[f'{mc}_ra']
            pct_null_error = (null_error / te_df[f'{mc}_ra']) * 100
            tmpdf = te_df.loc[:, ['centered_age']].copy()
            tmpdf['error'] = error
            emdl = smf.ols('error ~ centered_age', data=tmpdf)
            emdl_fitted = emdl.fit()
            res_age_terms = emdl_fitted.summary2().tables[1].loc['centered_age'].to_dict()
            tmpdf['ra'] = tmptedf[f'{mc}_ra']
            tmpdf['pred'] = pred
            tmpdf['orig'] = te_df[f'{mc}_{test_source}']
            mdlra = smf.ols('ra ~ centered_age', data=tmpdf)
            mdlra_fitted = mdlra.fit()
            mdlprd = smf.ols('pred ~ centered_age', data=tmpdf)
            mdlprd_fitted = mdlprd.fit()
            mdlorg = smf.ols('orig ~ centered_age', data=tmpdf)
            mdlorg_fitted = mdlorg.fit()
            
            mdl_res = dict(
                target='ra',
                train_source=train_source,
                test_source=test_source,
                metric=mc,
                mae_mdl=np.abs(error).mean(),
                mae_nomdl=np.abs(nomdl_error).mean(),
                mae_null=np.abs(null_error).mean(),
                mape_mdl=np.abs(pct_error).mean(),
                mape_nomdl=np.abs(pct_nomdl_error).mean(),
                mape_null=np.abs(pct_null_error).mean(),
                r2_mdl=1 - ((error**2).sum()/ ((null_error ** 2).sum())),
                r2_nomdl=1 - ((nomdl_error**2).sum()/ ((null_error ** 2).sum())),
                r2_mdl_v_nodml=1 - ((error**2).sum()/ ((nomdl_error ** 2).sum())),
                res_age_r2 = emdl_fitted.rsquared,
                ra_age_b = mdlra_fitted.params['centered_age'],
                ra_age_p = mdlra_fitted.pvalues['centered_age'],
                prd_age_b = mdlprd_fitted.params['centered_age'],
                prd_age_p = mdlprd_fitted.pvalues['centered_age'],
                org_age_b = mdlorg_fitted.params['centered_age'],
                org_age_p = mdlorg_fitted.pvalues['centered_age'],
            )
            #mdl_res.update(res_age_terms)
            mdl_resses.append(mdl_res)
mdl_resses=pd.DataFrame(mdl_resses)

In [ ]:
alpha = 0.05
mdl_resses['pct_prd_vs_ra_b'] = ((mdl_resses.prd_age_b - mdl_resses.ra_age_b) /  mdl_resses.ra_age_b) * 100
mdl_resses['pct_org_vs_ra_b'] = ((mdl_resses.org_age_b - mdl_resses.ra_age_b) /  mdl_resses.ra_age_b) * 100
mdl_resses['prd_TP'] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.prd_age_p < alpha)
mdl_resses['prd_TN'] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.prd_age_p > alpha)
mdl_resses['prd_FP'] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.prd_age_p < alpha)
mdl_resses['prd_FN'] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.prd_age_p > alpha)
mdl_resses['org_TP'] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.org_age_p < alpha)
mdl_resses['org_TN'] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.org_age_p > alpha)
mdl_resses['org_FP'] = (mdl_resses.ra_age_p > alpha) & (mdl_resses.org_age_p < alpha)
mdl_resses['org_FN'] = (mdl_resses.ra_age_p < alpha) & (mdl_resses.org_age_p > alpha)

In [ ]:
mdl_resses['delta_mape'] = mdl_resses.mape_mdl - mdl_resses.mape_nomdl

In [ ]:
mdl_resses.query("metric == 'G_and_S_frontomargin_lh'")

In [ ]:
mdl_resses.query("metric == 'G_and_S_occipital_inf_lh'")

In [ ]:
mdl_resses.groupby(['train_source', 'test_source']).prd_TP.mean()

In [ ]:
mdl_resses.groupby(['train_source', 'test_source']).org_TP.mean()

In [ ]:
mdl_resses.query('train_source == test_source').groupby('train_source').r2_mdl.describe()

In [ ]:
mdl_resses.query('train_source == test_source').groupby('train_source').delta_mape.describe()

In [ ]:
mdl_resses.query('train_source == "ract1"').groupby("test_source").r2_mdl.describe()

In [ ]:
mdl_resses.query('train_source == "ract1"').groupby("test_source").r2_mdl.describe()

In [ ]:
r2_num = ((error) ** 2).sum()
r2_den = 
r2 = 
r2

In [ ]:
r2_num = ((nomdl_error) ** 2).sum()
r2_den = (null_error ** 2).sum()
r2 = 1 - (r2_num/ r2_den)
r2

In [ ]:
r2_num = ((error) ** 2).sum()
r2_den = (nomdl_error ** 2).sum()
r2 = 1 - (r2_num/ r2_den)
r2

In [ ]:
plt.plot(te_df[f'{mc}_ra'], mdl_fitted.predict(te_df), 'o')

In [ ]:
plt.plot(te_df[f'{mc}_ra'], te_df[f'{mc}_ract1'], 'o')

In [ ]:
plt.plot(te_df.centered_age, te_df[f'{mc}_ra'] - mdl_fitted.predict(te_df), 'o')